## 1. CNN（Convolutional Neural Network）

`CNN`は、画像から**空間的な特徴を抽出して学習するニューラルネットワーク**である。

- **特徴**：畳み込みフィルタを使って、画像の輪郭・形・パターンなどを抽出する。
- **主な用途**：画像分類、物体検出、顔認識など
- **基本構造**：畳み込み層（特徴抽出） → プーリング層（特徴要約） → 全結合層（最終分類）
- **ANN / DNNとの違い**：CNNは、画像の空間的な特徴を効率よく学習できる。

例：猫の画像から耳・目・顔の形などの特徴を学習し、猫かどうかを分類する。

<br>

#### - MNISTデータセット

`mnist.load_data()`：Kerasが提供する**手書き数字データセットを読み込む関数**。

- 0～9の手書き数字画像
- 画像サイズ：**28 × 28（グレースケール）**
- 全データ：**70,000枚**
  - 学習用（Train）：60,000枚
  - テスト用（Test）：10,000枚

CNNなどの画像分類モデルを、簡単に動作確認・比較するための**基本的なベンチマークデータセット**としてよく使われる。

<br>

#### ※ CNNの代表的な流れ

<table style="
width:950px;
max-width:100%;
border-collapse:collapse;
color:#000;
text-align:left;
margin-left:0 !important;
margin-right:auto !important;
">
<tbody>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:7px 18px; width:280px; text-align:center; color:#000;">
<strong>画像入力</strong>
</td>
<td style="padding:7px 18px; color:#000;">
画像データを入力する。
</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td colspan="2" style="padding:5px 18px; text-align:center; color:#000;">↓</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:7px 18px; text-align:center; color:#000;">
<span style="color:#ff4d5a;"><strong>Conv2D</strong></span>
（畳み込み層・特徴抽出）
</td>
<td style="padding:7px 18px; color:#000;">
フィルタを使って、画像から輪郭・模様・形などの特徴を抽出する。
</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td colspan="2" style="padding:5px 18px; text-align:center; color:#000;">↓</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:7px 18px; text-align:center; color:#000;">
<span style="color:#ff4d5a;"><strong>MaxPooling2D</strong></span>
（最大プーリング層・特徴圧縮）
</td>
<td style="padding:7px 18px; color:#000;">
局所領域の最大値を取り、特徴を残しながら画像の空間サイズを縮小する。
</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td colspan="2" style="padding:5px 18px; text-align:center; color:#000;">↓</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:7px 18px; text-align:center; color:#000;">
<span style="color:#ff4d5a;"><strong>Dropout</strong></span>
（過学習抑制）
</td>
<td style="padding:7px 18px; color:#000;">
学習中に一部のユニットをランダムに0にし、過学習を抑える。
</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td colspan="2" style="padding:5px 18px; text-align:center; color:#000;">↓</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:7px 18px; text-align:center; color:#000;">
<span style="color:#ff4d5a;"><strong>Flatten</strong></span>
（平坦化）
</td>
<td style="padding:7px 18px; color:#000;">
抽出した多次元の特徴データを1次元に並べ、Dense層に渡せる形に変換する。
</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td colspan="2" style="padding:5px 18px; text-align:center; color:#000;">↓</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:7px 18px; text-align:center; color:#000;">
<span style="color:#ff4d5a;"><strong>Dense</strong></span>
（全結合層・分類）
</td>
<td style="padding:7px 18px; color:#000;">
抽出した特徴を組み合わせ、各クラスの予測に利用する。
</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td colspan="2" style="padding:5px 18px; text-align:center; color:#000;">↓</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:7px 18px; text-align:center; color:#000;">
<strong>予測結果</strong>
</td>
<td style="padding:7px 18px; color:#000;">
モデルが予測したクラスを出力する。
</td>
</tr>

</tbody>
</table>

<br>
<br>

## 【CNN画像分類の実習】

MNISTデータを用いて、CNNモデルの構築・学習設定・前処理・学習・評価を行う。

<br>

#### ① CNNモデル構築

上で確認したCNNの流れを、Kerasの `Sequential` を使って実装する。

<br>

In [2]:
from tensorflow.keras import Sequential
from tensorflow.keras import layers

model = Sequential([
    layers.Input(shape=(28, 28, 1)),          # 28×28・グレースケール画像

    layers.Conv2D(
        32, (3, 3), activation='relu'         # フィルタ32個・カーネル3×3
    ),
    layers.MaxPooling2D(),                    # 26×26 → 13×13
    layers.Dropout(0.1),                      # 10%をランダムに0にする

    layers.Conv2D(
        64, 3, activation='relu'              # フィルタ64個・カーネル3×3
    ),
    layers.MaxPooling2D(),                    # 11×11 → 5×5
    layers.Dropout(0.25),                     # 25%をランダムに0にする

    layers.Flatten(),                         # 5×5×64 → 1,600
    layers.Dense(128, activation='relu'),     # 全結合層：128ニューロン
    layers.Dropout(0.5),                      # 50%をランダムに0にする

    layers.Dense(10, activation='softmax')    # 0～9の10クラスを確率で出力
])

<br>

#### ② モデル構造の確認

`model.summary()`で、各層の**出力形状**と**パラメータ数**を確認する。

In [3]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

<br>

#### ※ `model.summary()` の見方

- **Output Shape**：各層を通過した後のデータ形状
- **Param #**：その層が学習するパラメータ数

<span style="color:red;">
※ MaxPooling2D・Dropout・Flattenは学習可能なパラメータを持たないため、このモデルでは Param # は0となる。<br>
<span style="padding-left:1.2em;">Dense(128) は204,928個のパラメータを持ち、このモデルの中で最も多い。</span>
</span>

<br>

<br>

#### ③ 学習設定

MNISTは **0～9の10クラス分類**で、正解ラベルは整数で与えられる。

In [4]:
model.compile(
    loss='sparse_categorical_crossentropy',  # 整数ラベルを使う多クラス分類
    optimizer='adam',                        # 重みを更新
    metrics=['accuracy']                     # 正解率を確認
)

<br>

#### ④ MNISTデータ読み込み

`mnist.load_data()`を使って、学習用・テスト用の画像と正解ラベルを読み込む。

In [5]:
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

display(X_train[0])   # 最初の28×28画像
display(y_train[0])   # 最初の正解ラベル

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


array([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   3,
         18,  18,  18, 126, 136, 175,  26, 166, 255, 247, 127,   0,   0,
          0,   0],
       [  

np.uint8(5)

<br>

#### ⑤ データ形状確認

MNISTの画像データと正解ラベルの形を確認する。

In [6]:
display(X_train.shape)  # (60000, 28, 28)
display(y_train.shape)  # (60000,)

display(X_test.shape)   # (10000, 28, 28)
display(y_test.shape)   # (10000,)

(60000, 28, 28)

(60000,)

(10000, 28, 28)

(10000,)

<br>

#### ⑥ 画像データ前処理

CNNに入力できる形に変換し、ピクセル値を **0～1** に正規化する。

- MNIST画像は `(28, 28)` の2次元データだが、Conv2Dでは  
**高さ × 幅 × チャンネル** の3次元形式を使うため、  
グレースケール画像にはチャンネル `1` を追加する。

In [7]:
X_train = X_train.reshape(-1, 28, 28, 1) / 255.0
X_test = X_test.reshape(-1, 28, 28, 1) / 255.0

# (28, 28) → (28, 28, 1)：グレースケールのチャンネルを追加
# 0～255 → 0～1：ピクセル値を正規化
# -1：画像数を自動計算

<br>

#### ⑦ モデル学習

前処理したMNIST画像を使ってCNNモデルを学習する。

In [8]:
model.fit(
    X_train,
    y_train,
    batch_size=128,  # 1回に128枚ずつ学習
    epochs=10,       # 全学習データを10回繰り返す
    verbose=1        # 学習経過を表示
)

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 18s 35ms/step - accuracy: 0.8950 - loss: 0.3336
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 18s 39ms/step - accuracy: 0.9677 - loss: 0.1084
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 19s 39ms/step - accuracy: 0.9751 - loss: 0.0826
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 17s 35ms/step - accuracy: 0.9799 - loss: 0.0661
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 14s 31ms/step - accuracy: 0.9830 - loss: 0.0577
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.9836 - loss: 0.0554
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - accuracy: 0.9853 - loss: 0.0482
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - accuracy: 0.9868 - loss: 0.0449
Epoch 9/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.9863 - loss: 0.0427
Epoch 10/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 14s 31ms/step - accuracy: 0.9886 - loss: 0.0391


<Br>

#### ⑧ モデル評価

学習に使用していないテストデータで、モデルの分類性能を確認する。

- **Loss**：予測と正解のずれ → **小さいほど良い**
- **Accuracy**：正しく分類できた割合 → **高いほど良い**

In [9]:
loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0  # 評価経過を非表示
)

print(f"Test loss：{loss:.4f}")
print(f"Test accuracy：{accuracy:.4f}")

Test loss：0.0242
Test accuracy：0.9917


<br>

#### ※ CNN画像分類の実習フロー

<table align="left" style="
width:950px;
max-width:100%;
border-collapse:collapse;
color:#000;
text-align:left;
margin-left:0 !important;
margin-right:auto !important;
">

<thead>
<tr style="background-color:#dfe3e8; color:#000;">
<th style="padding:8px 16px; width:70px; text-align:center;"><strong>段階</strong></th>
<th style="padding:8px 16px; width:280px; text-align:center;"><strong>作業</strong></th>
<th style="padding:8px 16px; text-align:center;"><strong>内容</strong></th>
</tr>
</thead>

<tbody>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>①</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>CNNモデル構築</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>Sequential</code>でCNNの各層を構成する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>②</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>モデル構造確認</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>model.summary()</code>で出力形状・パラメータ数を確認する。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>③</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>学習設定</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>compile()</code>で損失関数・最適化手法・評価指標を設定する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>④</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>MNISTデータ読み込み</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>mnist.load_data()</code>で学習用・テスト用データを読み込む。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑤</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>データ形状確認</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>shape</code>で画像数・画像サイズ・ラベル数を確認する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑥</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>画像データ前処理</strong></td>
<td style="padding:8px 16px; text-align:left;">チャンネルを追加し、ピクセル値を0～1に正規化する。</td>
</tr>

<tr style="background-color:#e9ecef; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑦</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>モデル学習</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>fit()</code>で学習データを使ってCNNを学習する。</td>
</tr>

<tr style="background-color:#f8f8f8; color:#000;">
<td style="padding:8px 16px; text-align:center;"><strong>⑧</strong></td>
<td style="padding:8px 16px; text-align:center;"><strong>モデル評価</strong></td>
<td style="padding:8px 16px; text-align:left;"><code>evaluate()</code>でテストデータのLoss・Accuracyを確認する。</td>
</tr>

</tbody>
</table>

<br clear="all">